# Customer Service

> Notebook generated from [`examples/subgraphs/06_customer_service.py`](../../examples/subgraphs/06_customer_service.py).

| Item | Value |
|------|-------|
| **Dataset** | ATIS + Amazon Reviews (embedded) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** classifier → faq_retrieval [RAG] → escalation_gate → response_generator | ticket_creator. Per ticket: branch taken.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Customer Service Subgraph — Customer support with RAG and escalation
=====================================================================
Subgraph: prismal.agents.subgraphs.customer_service

Dataset: ATIS + Amazon Customer Reviews (simulated support)
  • ATIS (Airline Travel Information System): 5,000+ user queries
    in natural language classified by intent (flight, fare, ground,
    abbreviation, capacity, city, …).
  • Amazon Customer Reviews: real technical-support / return queries.
  • Reference: https://huggingface.co/datasets/atis and
    https://huggingface.co/datasets/amazon_reviews_multi
  • Why: The customer_service subgraph classifies intents (faq /
    complaint / technical / other) and routes by confidence. ATIS provides
    clean intents; Amazon Reviews adds complaints and ambiguous cases that
    exercise the escalation_gate.

Customer Service subgraph description:
  classifier → faq_retrieval → escalation_gate ─┬→ response_generator
                                                 └→ ticket_creator

  Nodes:
  1. classifier        — LLM classifies the query as faq/complaint/technical/other
  2. faq_retrieval     — RAG over the internal knowledge base
  3. escalation_gate   — if confidence < threshold → ticket_creator
                         if confidence ≥ threshold → response_generator
  4. response_generator — drafts the final response to the customer
  5. ticket_creator    — opens a support ticket and returns confirmation

  escalation_threshold: 0.6 (default) — below → escalates to ticket

Usage:
    uv run python examples/subgraphs/06_customer_service.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import random

# Import with error handling
try:
    from prismal.agents.subgraphs.customer_service.builder import (
        build_customer_service_subgraph,
        register_customer_service,
    )
    from prismal.agents.subgraphs.factory import assemble_state_graph

    CUSTOMER_SERVICE_AVAILABLE = True
except ImportError:
    CUSTOMER_SERVICE_AVAILABLE = False

## Dataset: support queries

In [ ]:
SUPPORT_QUERIES = [
    # Category: FAQ (should be routed to response_generator)
    {
        "id": "CS-001",
        "query": "What is your return policy?",
        "category": "faq",
        "expected_route": "response_generator",
        "context": "Customer asks about return deadlines and conditions.",
        "user_id": "U1001",
    },
    {
        "id": "CS-002",
        "query": "How can I track my order number 847392?",
        "category": "faq",
        "expected_route": "response_generator",
        "context": "Standard shipping tracking inquiry.",
        "user_id": "U1002",
    },
    {
        "id": "CS-003",
        "query": "What are your business hours and support channels?",
        "category": "faq",
        "expected_route": "response_generator",
        "context": "Question about support hours.",
        "user_id": "U1003",
    },
    # Category: Technical (high confidence → response_generator)
    {
        "id": "CS-004",
        "query": "The product I received does not power on, what should I do?",
        "category": "technical",
        "expected_route": "response_generator",
        "context": "Technical issue with newly received electronic device.",
        "user_id": "U1004",
    },
    {
        "id": "CS-005",
        "query": "How do I reset my password? I've been locked out.",
        "category": "technical",
        "expected_route": "response_generator",
        "context": "User account access blocked.",
        "user_id": "U1005",
    },
    # Category: Complaint (low confidence → ticket_creator)
    {
        "id": "CS-006",
        "query": "This is unacceptable. I've been waiting 3 weeks for my refund and "
        "nobody is answering. I'm going to leave a negative review everywhere.",
        "category": "complaint",
        "expected_route": "ticket_creator",
        "context": "Very frustrated customer, urgent escalation case.",
        "user_id": "U1006",
    },
    {
        "id": "CS-007",
        "query": "I was charged twice for the same order. I need my money refunded "
        "urgently. This is fraud.",
        "category": "complaint",
        "expected_route": "ticket_creator",
        "context": "Double charge on bank account, requires human intervention.",
        "user_id": "U1007",
    },
    # Category: Other/Ambiguous (variable confidence)
    {
        "id": "CS-008",
        "query": "I want to know if you can make an exception to the return policy "
        "for my particular case, since the gift was for my sick mother and "
        "she couldn't use it.",
        "category": "other",
        "expected_route": "ticket_creator",
        "context": "Exception request — requires human decision.",
        "user_id": "U1008",
    },
]

## FAQ Knowledge Base

In [ ]:
# In production this would be indexed in ChromaVectorStore.
FAQ_KB = [
    {
        "id": "faq_001",
        "question": "What is the return policy?",
        "answer": (
            "We accept returns within 30 days from the purchase date. "
            "The product must be in its original condition with all accessories. "
            "Refunds are processed within 3-5 business days after we receive the item."
        ),
        "category": "returns",
    },
    {
        "id": "faq_002",
        "question": "How do I track my order?",
        "answer": (
            "You can track your order under 'My Account → Orders'. You will also receive "
            "an email with the carrier's tracking number once "
            "the package is shipped."
        ),
        "category": "shipping",
    },
    {
        "id": "faq_003",
        "question": "What are the customer service hours?",
        "answer": (
            "Our support team is available Monday to Friday from 9:00 to 18:00 "
            "(CET). You can also reach us by email at support@company.com with "
            "a response in under 24 hours."
        ),
        "category": "support",
    },
    {
        "id": "faq_004",
        "question": "My device does not power on?",
        "answer": (
            "1. Make sure the battery is fully charged (at least 2h). "
            "2. Hold the power button for 10 seconds. "
            "3. If it still doesn't work, it may be a hardware failure. "
            "Contact technical support attaching a photo of the defect."
        ),
        "category": "technical",
    },
    {
        "id": "faq_005",
        "question": "How do I recover my password?",
        "answer": (
            "Go to 'Sign in → Forgot your password?' and enter your email. "
            "You'll receive a recovery link valid for 24 hours. If you don't get it, "
            "check your spam folder or contact support@company.com."
        ),
        "category": "account",
    },
]

## Pipeline simulator

In [ ]:
def classify_intent(query: str) -> tuple[str, float]:
    """Simulate the classifier node (LLM in real mode).

    Returns:
        (category, confidence) — category: faq/complaint/technical/other
    """
    query_lower = query.lower()

    # Complaint signals
    complaint_signals = [
        "unacceptable",
        "fraud",
        "urgent",
        "never",
        "negative review",
        "three weeks",
        "charged twice",
        "refunded",
        "scandal",
    ]
    faq_signals = [
        "policy",
        "hours",
        "track",
        "tracking",
        "how can",
        "how do i",
        "what are",
        "business hours",
        "password",
        "reset",
    ]
    technical_signals = [
        "does not power on",
        "does not work",
        "error",
        "failure",
        "blocked",
        "locked out",
        "doesn't work",
    ]

    complaint_score = sum(1 for s in complaint_signals if s in query_lower)
    faq_score = sum(1 for s in faq_signals if s in query_lower)
    technical_score = sum(1 for s in technical_signals if s in query_lower)

    if complaint_score >= 2:
        return "complaint", 0.30  # low confidence → will escalate
    if complaint_score == 1:
        return "complaint", 0.45  # still low → will escalate
    if technical_score >= 1:
        return "technical", 0.75
    if faq_score >= 1:
        return "faq", 0.85
    return "other", 0.40  # ambiguous → will escalate


def faq_retrieval(query: str, top_k: int = 1) -> tuple[str | None, float]:
    """Simulate the faq_retrieval node (RAG in real mode).

    Returns:
        (answer, confidence) — None if no match is found
    """
    query_lower = query.lower()
    best_score = 0.0
    best_answer = None

    for faq in FAQ_KB:
        # Keyword overlap (proxy for semantic similarity)
        faq_text = (faq["question"] + " " + faq["answer"]).lower()
        query_words = set(query_lower.split())
        faq_words = set(faq_text.split())
        overlap = len(query_words & faq_words) / max(len(query_words), 1)

        if overlap > best_score:
            best_score = overlap
            best_answer = faq["answer"]

    # Scale score to range [0.3, 0.9]
    confidence = min(0.3 + best_score * 3.0, 0.9)
    return (best_answer, confidence) if best_score > 0.05 else (None, 0.0)


def generate_ticket_id(user_id: str) -> str:
    """Generate a support ticket ID."""
    ticket_num = random.randint(100000, 999999)
    return f"TKT-{ticket_num}"


def generate_response(query: str, faq_answer: str | None, category: str) -> str:
    """Simulate the response_generator node."""
    if faq_answer:
        return (
            f"Hi, thanks for reaching out. {faq_answer} Is there anything else I can help with?"
        )
    templates = {
        "faq": "Hi, thanks for your inquiry. I've reviewed your case and can confirm that our team will assist you in the next few hours.",
        "technical": "Hi, sorry for the inconvenience. To solve your technical issue, please follow these steps: 1) Restart the device. 2) If it persists, email us at support@company.com with a photo.",
        "other": "Hi, we have received your inquiry. A specialized agent will review your case and contact you within 24 hours.",
    }
    return templates.get(category, "Thank you for contacting us. An agent will help you soon.")


async def run_customer_service(query_data: dict, escalation_threshold: float = 0.6) -> dict:
    """Run the customer service pipeline for a query."""
    query = query_data["query"]
    print(f"\n[{query_data['id']}] {query_data['category'].upper()}")
    print(f"  Query: {query[:80]}{'...' if len(query) > 80 else ''}")
    print(f"  User : {query_data['user_id']}")

    if not CUSTOMER_SERVICE_AVAILABLE:
        print("  [Demo mode — simulated subgraph]")

        # Node 1: classifier
        category, cls_confidence = classify_intent(query)
        print("\n  ── Node 1: classifier ──")
        print(f"    Category   : {category}")
        print(f"    Confidence : {cls_confidence:.2f}")

        # Node 2: faq_retrieval
        print("\n  ── Node 2: faq_retrieval ──")
        faq_answer, faq_confidence = faq_retrieval(query)
        overall_confidence = (cls_confidence + faq_confidence) / 2
        print(f"    FAQ found       : {'Yes' if faq_answer else 'No'}")
        print(f"    RAG confidence  : {faq_confidence:.2f}")
        print(f"    Total confidence: {overall_confidence:.2f}")
        print(f"    Escalation threshold: {escalation_threshold}")

        # Gate: escalation
        print("\n  ── Gate: escalation_gate ──")
        should_escalate = overall_confidence < escalation_threshold or category == "complaint"
        route = "ticket_creator" if should_escalate else "response_generator"
        print(f"    Escalate? {'Yes' if should_escalate else 'No'}  →  {route}")

        result: dict = {
            "id": query_data["id"],
            "category": category,
            "confidence": overall_confidence,
            "route": route,
        }

        if should_escalate:
            # Node 5: ticket_creator
            print("\n  ── Node 5: ticket_creator ──")
            ticket_id = generate_ticket_id(query_data["user_id"])
            print(f"    Ticket created: {ticket_id}")
            print(f"    Priority     : {'High' if category == 'complaint' else 'Normal'}")
            print("    Assigned to  : human_support")
            result["ticket_id"] = ticket_id
            result["response"] = (
                f"We have logged your case with ticket number {ticket_id}. "
                f"A specialized agent will contact you within 2-4 hours. "
                f"Sorry for the inconvenience."
            )
        else:
            # Node 4: response_generator
            print("\n  ── Node 4: response_generator ──")
            response = generate_response(query, faq_answer, category)
            print(f"    Response generated ({len(response)} chars)")
            result["response"] = response

        print(f"\n  Response: {result['response'][:100]}...")

        # Check whether the route matches the expected one
        expected = query_data["expected_route"]
        match = "✓" if route == expected else "✗"
        print(f"  Route: {route} {match} (expected: {expected})")

        return result

    # Real mode with LangGraph subgraph
    from langchain_core.messages import HumanMessage

    from prismal.agents.state import create_initial_state

    await register_customer_service(escalation_threshold=escalation_threshold)
    subgraph = build_customer_service_subgraph(
        escalation_threshold=escalation_threshold,
    )
    graph = assemble_state_graph(subgraph).compile()

    state = create_initial_state(session_id="nb-customer-service")
    state["messages"] = [HumanMessage(content=query)]
    state["metadata"] = {
        "customer_service": {
            "user_id": query_data["user_id"],
            "query_id": query_data["id"],
        }
    }

    config = {"configurable": {"thread_id": f"cs_{query_data['id']}_001"}}
    final_state = await graph.ainvoke(state, config=config)

    cs_meta = final_state.get("metadata", {}).get("customer_service", {})
    messages = final_state.get("messages", [])
    return {
        "id": query_data["id"],
        "route": cs_meta.get("route", "unknown"),
        "response": str(messages[-1].content) if messages else "",
        "ticket_id": cs_meta.get("ticket_id"),
    }


async def main() -> None:
    ESCALATION_THRESHOLD = 0.6

    print("=" * 70)
    print("  Customer Service Subgraph — Dataset: ATIS + Amazon Reviews")
    print("=" * 70)

    print("\n[Customer Service subgraph architecture]")
    print("  classifier")
    print("       ↓")
    print("  faq_retrieval  ←  RAG over Knowledge Base")
    print("       ↓")
    print("  escalation_gate")
    print(f"       ├── confidence ≥ {ESCALATION_THRESHOLD} → response_generator → customer")
    print(f"       └── confidence < {ESCALATION_THRESHOLD} → ticket_creator → human agent")

    print(f"\n[Knowledge Base: {len(FAQ_KB)} FAQs available]")
    for faq in FAQ_KB:
        print(f"  [{faq['category']:10s}] {faq['question']}")

    print(f"\n[Processing {len(SUPPORT_QUERIES)} support queries]")
    results = []
    for query_data in SUPPORT_QUERIES:
        result = await run_customer_service(query_data, ESCALATION_THRESHOLD)
        results.append(result)
        print("─" * 70)

    # ── Statistics ────────────────────────────────────────────────────────────
    print("\n[Statistical summary]")

    routed_to_response = sum(1 for r in results if r["route"] == "response_generator")
    routed_to_ticket = sum(1 for r in results if r["route"] == "ticket_creator")
    correct_routes = sum(
        1
        for r, q in zip(results, SUPPORT_QUERIES, strict=False)
        if r["route"] == q["expected_route"]
    )

    print(f"  Queries processed     : {len(results)}")
    print(
        f"  → response_generator  : {routed_to_response} ({routed_to_response / len(results):.0%})"
    )
    print(f"  → ticket_creator      : {routed_to_ticket} ({routed_to_ticket / len(results):.0%})")
    print(
        f"  Correct routes        : {correct_routes}/{len(results)} "
        f"({correct_routes / len(results):.0%} routing accuracy)"
    )
    tickets = [r.get("ticket_id") for r in results if r.get("ticket_id")]
    if tickets:
        print(f"  Tickets created       : {tickets}")

    # ── Threshold comparison ──────────────────────────────────────────────────
    print("\n[Impact of escalation_threshold on routing]")
    print(f"  {'Threshold':<12} {'→ Response':<14} {'→ Ticket':<12} {'Description'}")
    print("  " + "─" * 60)
    thresholds = [
        (0.3, "Almost everything auto-resolved, little escalation"),
        (0.6, "Optimal balance (default)"),
        (0.8, "Aggressive escalation, more human tickets"),
        (1.0, "Everything escalates — no auto-response"),
    ]
    for thresh, desc in thresholds:
        # Simulate counts with the given threshold
        sim_auto = sum(
            1
            for q in SUPPORT_QUERIES
            if classify_intent(q["query"])[1] >= thresh and q["category"] != "complaint"
        )
        sim_ticket = len(SUPPORT_QUERIES) - sim_auto
        marker = "← recommended" if thresh == 0.6 else ""
        print(f"  {thresh:<12.1f} {sim_auto:<14d} {sim_ticket:<12d} {desc} {marker}")

    print("\n[When to use Customer Service Subgraph]")
    print("  ✓ Companies with high volume of repetitive queries (FAQ)")
    print("  ✓ When there is a well-curated Knowledge Base")
    print("  ✓ To reduce time to first response (TTFR)")
    print("  ✓ Smart escalation to human agents for complex cases")
    print("  ✗ Without a Knowledge Base → will always escalate (RAG confidence = 0)")


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()